In [ ]:
import torch
import torch.nn as nn
torch.manual_seed(42)

# 평균제곱오차
$$
MSE = \frac{1}{n}\sum_{i=1}^{n}(y_i-\hat{y}_i)^2
$$

In [ ]:
# 예측값과 정답값 (회귀)
pred = torch.tensor([2.5, 5.0, 7.5])
target = torch.tensor([3.0, 5.0, 7.0])

# pytorch에서 제공하는 MSELoss 함수
mse_buildin = nn.MSELoss()(pred, target)

# 수식 기반
mse_manual = torch.mean((pred - target)**2)

print('mse_buildin:',mse_buildin)
print('mse_manual:',mse_manual)

# 이진분류 손실
$$
BCE = -\frac{1}{n}\sum_{i=1}^{n}\left[y_i\log(\hat{y}_i)+(1-y_i)\log(1-\hat{y}_i)\right]
$$


In [ ]:
# 단일샘플 y = 1 일 때 식은 -log(y_hat) 단순화 / 정답 확률이 높을수록 손실이 낮아지는 형태
y = torch.tensor([1.0])
y_hat_good = torch.tensor([0.9])  # 정답확률 높음
y_hat_bad = torch.tensor([0.1])   # 정답확률 낮음
# 수작업 계산
bce_good_manual = -(y*torch.log(y_hat_good) + (1 - y) * torch.log(1 - y_hat_good))
bce_bad_manual = -(y*torch.log(y_hat_bad) + (1 - y) * torch.log(1 - y_hat_bad))

# 내장된 bce
bce_good_builtin = nn.BCELoss()(y_hat_good, y)
bce_bad_builtin = nn.BCELoss()(y_hat_bad, y)

print(f'bce_good_manual : {bce_good_manual}')
print(f'bce_bad_manual : {bce_bad_manual}')
print(f'bce_good_builtin : {bce_good_builtin}')
print(f'bce_bad_builtin : {bce_bad_builtin}')

In [ ]:
# bce 배치계산
prob_pred = torch.tensor([0.9, 0.2, 0.4])
label = torch.tensor([1.0, 0.0, 1.0])

loss = nn.BCELoss()(prob_pred, label)
print(loss)

# 다중분류 손실
$$
CE = -\sum_{i=1}^{m} y_i\log(\hat{y}_i)
$$

In [ ]:
# 클래스 3개 [고양이, 강아지, 토끼] -> 정답 강아지 (index=1)
target_index = torch.tensor([1])
one_hot = torch.tensor([[0.0, 1.0, 0.0]]) # 원-핫 인코딩된 정답

# 좋은 예측과 나쁜 예측
p_good = torch.tensor([0.2, 0.7, 0.1])
p_bad = torch.tensor([0.7, 0.2, 0.1])

# 수작업
ce_good_manual = -(one_hot*torch.log(p_good)).sum()
ce_bad_manual = -(one_hot*torch.log(p_bad)).sum()

# torch bce : logits 입력 log(p)
# unsqueeze : 차수(차원 수)를 추가
ce_loss = nn.CrossEntropyLoss()
ce_good_builtin = ce_loss(torch.log(p_good).unsqueeze(0), target_index)
ce_bad_builtin = ce_loss(torch.log(p_bad).unsqueeze(0), target_index)

print(ce_good_manual, ce_good_builtin)
print(ce_bad_manual, ce_bad_builtin)

# 실제 데이터에 적용

In [ ]:
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
# 회귀 모델
X, y = load_diabetes(return_X_y=True)

scaler = StandardScaler()
x_train, x_test, y_train, y_test = train_test_split(X, y, random_state=42)
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

x_train_t = torch.tensor(x_train, dtype=torch.float32)
x_test_t = torch.tensor(x_test, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
y_test_t = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

reg_model = nn.Sequential(
    nn.Linear(x_train_t.shape[1], 64),     # 은닉층
    nn.ReLU(),                             # 활성화 함수
    nn.Linear(64, 1)                       # 출력층
)

mse_loss = nn.MSELoss()
optimizer = torch.optim.Adam(reg_model.parameters(), lr=1e-2)  # 1e-2 : 0.01

epochs = 300
for _ in range(epochs):
    optimizer.zero_grad() # 이전 가중치 초기화
    logits = reg_model(x_train_t)
    loss = mse_loss(logits, y_train_t)
    loss.backward()      # backward : 가중치 찾아서 각 계산과정에 배치
    optimizer.step()     # 각 계산과정의 가중치와 바이어스 업데이트

with torch.no_grad():
    test_pred = reg_model(x_test_t)
    test_mse = mse_loss(test_pred, y_test_t)

# test_mse -> 텐서에 들어있음 / 출력이나 기타 산술연산이 필요 / 꺼내야함 : items()
print(f'{test_mse.item():.4f}')

from sklearn.metrics import r2_score

r2_score(y_test_t, test_pred)

In [ ]:
from sklearn.datasets import load_breast_cancer

# 1. 딥러닝 레이어 Sequential에서 입력받는 부분 수정 (피쳐개수)
# 2. 출력 레이어의 활성화 함수로 시그모이드
# 3. 손실함수 nn.BCELoss()
# 추론 with -> 0.5보다 크면 1 그렇지 않으면 0 변경

# 분류모델 (이진분류)
X, y = load_breast_cancer(return_X_y=True)

scaler = StandardScaler()
x_train, x_test, y_train, y_test = train_test_split(X, y, random_state=42)
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

x_train_t = torch.tensor(x_train, dtype=torch.float32)
x_test_t = torch.tensor(x_test, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
y_test_t = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

clf_model = nn.Sequential(
    nn.Linear(x_train_t.shape[1], 64),     # 은닉층
    nn.ReLU(),                             # 활성화 함수
    nn.Linear(64, 1),                      # 출력층
    nn.Sigmoid()
)

bce_loss = nn.BCELoss()
optimizer = torch.optim.Adam(clf_model.parameters(), lr=1e-2)  # 1e-2 : 0.01

epochs = 300
for _ in range(epochs):
    optimizer.zero_grad() # 이전 가중치 초기화
    logits = clf_model(x_train_t)
    loss = bce_loss(logits, y_train_t)
    loss.backward()      # backward : 가중치 찾아서 각 계산과정에 배치
    optimizer.step()     # 각 계산과정의 가중치와 바이어스 업데이트

with torch.no_grad():
    test_pred = clf_model(x_test_t)
    test_pred_label = (test_pred >= 0.5).float()

from sklearn.metrics import accuracy_score

acc= accuracy_score(y_test, test_pred_label.numpy())
print(acc)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, confusion_matrix, ConfusionMatrixDisplay

# 시각화
with torch.no_grad():
    test_pred = clf_model(x_test_t)
    test_prob = test_pred.numpy().ravel()

fpr, tpr, thresholds = roc_curve(y_test, test_prob)
roc_auc = auc(fpr, tpr)

plt.figure()
plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.4f}")
plt.plot([0, 1], [0, 1], linestyle="--")  # 랜덤 기준선
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")

cm = confusion_matrix(y_test, test_pred_label)

disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot()
plt.title("Confusion Matrix")

plt.legend()
plt.show()

In [ ]:
from sklearn.datasets import load_iris

# 신경망의 입력데이터 개수
# 신경망의 최종 출력의 뉴런 수 3
# 최종 출력의 마지막에 활성함수 softmax 사용안함
# 손실함수 nn.CrossEntropyLoss()

# 분류모델 (다중분류)
X, y = load_iris(return_X_y=True)

scaler = StandardScaler()
x_train, x_test, y_train, y_test = train_test_split(X, y, random_state=42)
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

x_train_t = torch.tensor(x_train, dtype=torch.float32)
x_test_t = torch.tensor(x_test, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
y_test_t = torch.tensor(y_test, dtype=torch.long)

clf_model = nn.Sequential(
    nn.Linear(x_train_t.shape[1], 64),     # 은닉층
    nn.ReLU(),                             # 활성화 함수
    nn.Linear(64, 3)                       # 출력층
)

ce_loss = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(clf_model.parameters(), lr=1e-2)  # 1e-2 : 0.01

epochs = 300
for _ in range(epochs):
    optimizer.zero_grad() # 이전 가중치 초기화
    logits = clf_model(x_train_t)
    loss = ce_loss(logits, y_train_t)
    loss.backward()      # backward : 가중치 찾아서 각 계산과정에 배치
    optimizer.step()     # 각 계산과정의 가중치와 바이어스 업데이트

with torch.no_grad():
    test_pred = clf_model(x_test_t)
    test_pred_label = torch.argmax(test_pred, dim=1)

from sklearn.metrics import accuracy_score

acc= accuracy_score(y_test, test_pred_label.numpy())
print(acc)